# Lab 09: From a bird's-eye-view grid to camera pixels (BEVFormer geometry)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hokwangchoi/hc-model-study/blob/master/labs/notebooks/lab09_bev_projection.ipynb) · lab page: [`labs/lab09.html`](../lab09.html) · file: [`labs/lab09_bev_projection.py`](../lab09_bev_projection.py)

**Time:** 60-90 min

## Goal

A BEV model builds one top-down feature map around the car from several
cameras. The geometric core is plain linear algebra: lift every BEV cell to
a few 3-D points, project them into each camera, keep the ones that land
inside an image, and bilinearly sample the image features there. You build
camera intrinsics and extrinsics, the projection, the BEV pillar points,
the reference points with their visibility mask, and the feature sampling
with F.grid_sample.

## The maths, step by step

```text
Frames:  ego     x forward, y left, z up (origin on the ground under the car)
         camera  x right,   y down, z forward = depth (OpenCV convention)

  1. Intrinsics K (pinhole):   [[f, 0, cx], [0, f, cy], [0, 0, 1]],  cx = W/2, cy = H/2
  2. Extrinsics ego -> camera for a camera at height h, turned by yaw psi
     (counter-clockwise from +x, towards +y = left):
       forward  z_c = ( cos psi,  sin psi,  0)     (optical axis, in ego coords)
       right    x_c = ( sin psi, -cos psi,  0)
       down     y_c = ( 0,        0,       -1)
       R = [x_c; y_c; z_c]  (rows),  C = (0, 0, h)  camera centre
       p_cam = R (p_ego - C) = R p_ego + t,  t = -R C
     stored as a 4 x 4 matrix [[R, t], [0, 0, 0, 1]].
  3. Projection:  (X, Y, Z) = p_cam;   u = f X / Z + cx,   v = f Y / Z + cy
     Z is the depth; a point with Z <= 0 is behind the camera.
  4. BEV pillars: cell (i, j) of a bev_h x bev_w grid with resolution r metres
       x = (i - (bev_h - 1) / 2) * r     (forward)
       y = (j - (bev_w - 1) / 2) * r     (left)
     lifted to N_z heights z_1..z_Nz. Query index q = i * bev_w + j.
       points: [Q = bev_h * bev_w, N_z, 3]
  5. Reference points and mask, per camera:
       uv_norm = (2u / W - 1, 2v / H - 1)          in [-1, 1] inside the image
       valid   = (Z > eps) and (0 < u < W) and (0 < v < H)
     ([-1, 1] coordinates are resolution-free: the same uv_norm samples the
     image or a stride-16 feature map of it.)
  6. Sampling: F.grid_sample(features [N_cam, C, Hf, Wf], uv_norm as grid
     [N_cam, Q, N_z, 2], bilinear, align_corners=False) -> [N_cam, C, Q, N_z].
     BEVFormer's spatial cross-attention then averages over the cameras that
     see the cell (and mixes the heights with learned attention weights);
     here a plain average over the valid cameras and heights stands in for it.

  Worked example (guide, front camera, W x H = 800 x 448, f = 500, h = 1.6 m):
     p = (10, 2, 0) -> p_cam = (-2, 1.6, 10) -> (u, v) = (300, 304)
     -> uv_norm = (-0.25, 0.357). The rear camera sees Z = -10: invalid.
```

## Read alongside

- [BEV guide](../../bev/bev_guide.html): "The Core Problem: 6 Cameras -> One Top-Down Map", 3. Camera Projection (worked example, #proj-widget), 4. Deformable Spatial Cross-Attention
- [bev.py](../../bev/bev.py): make_camera_rig, project_ego_to_image, make_bev_pillar_points

## How to work

1. Run the setup cell and the imports cell.
2. Implement the functions in order, replacing `raise NotImplementedError  # TODO`. The hints above each cell open one at a time; try Hint 1 first.
3. Run each cell after editing it, then run the check cell at the end: `check.run_lab(9, namespace=globals())` checks the functions defined in this notebook (TODO = not started, PASS = matches the reference, FAIL = read the message).

Your work stays in this notebook; it does not change `labs/lab09_bev_projection.py` or the lab page. The reference solution is `labs/solutions/lab09_bev_projection.py`.

In [ ]:
# Setup: make the checker (labs/check.py) importable. Run this cell first.
# In a clone of the repository (Jupyter, VS Code) it uses labs/check.py;
# elsewhere (e.g. Google Colab) it downloads check.py, which needs only torch.
import pathlib
import sys
import urllib.request


def _find_labs_dir():
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        for cand in (d, d / "labs"):
            if (cand / "check.py").is_file() and (cand / "solutions").is_dir():
                return cand
    return None


LABS_DIR = _find_labs_dir()
if LABS_DIR is None:
    LABS_DIR = pathlib.Path("hc_model_study_labs").resolve()
    LABS_DIR.mkdir(exist_ok=True)
    if not (LABS_DIR / "check.py").is_file():
        urllib.request.urlretrieve("https://raw.githubusercontent.com/hokwangchoi/hc-model-study/master/labs/check.py", LABS_DIR / "check.py")
sys.path.insert(0, str(LABS_DIR))
import check  # noqa: E402

import torch  # noqa: E402
print("checker:", LABS_DIR / "check.py", "| torch", torch.__version__)

### Imports

In [ ]:
import torch
# You will need two more imports: math (cos, sin, radians) and
# torch.nn.functional as F (F.grid_sample).

## Step 1: intrinsics and extrinsics

**Hints for `make_intrinsics`** (open one at a time):

<details><summary>Hint 1</summary>

```text
a 3 x 3 torch.tensor; write 0.0 rather than 0 so the dtype is float32.
```

</details>

In [ ]:
def make_intrinsics(f: float, img_w: int, img_h: int) -> torch.Tensor:
    """K [3, 3] float32 with focal length f (pixels) and principal point at the image centre."""
    raise NotImplementedError  # TODO

**Hints for `make_ego_to_cam`** (open one at a time):

<details><summary>Hint 1</summary>

```text
build the three camera axes in ego coordinates (see the docstring), stack
them as ROWS of R: then R @ p expresses p in camera axes.
```

</details>

<details><summary>Hint 2</summary>

```text
t = -R @ C with C = (0, 0, cam_height).
```

</details>

<details><summary>Hint 3</summary>

```text
T = torch.eye(4); T[:3, :3] = R; T[:3, 3] = t
```

</details>

In [ ]:
def make_ego_to_cam(yaw_deg: float, cam_height: float) -> torch.Tensor:
    """[4, 4] float32 ego -> camera transform for a camera at (0, 0, cam_height) facing yaw_deg."""
    raise NotImplementedError  # TODO

## Step 2: project 3-D points into one camera

**Hints for `project_points`** (open one at a time):

<details><summary>Hint 1</summary>

```text
homogeneous coordinates: append a column of ones -> [P, 4], then
p_cam = (ego_to_cam @ p_h.T).T[:, :3]   (or p_h @ ego_to_cam.T).
```

</details>

<details><summary>Hint 2</summary>

```text
pix = p_cam @ K.T gives (f X + cx Z, f Y + cy Z, Z) per row.
```

</details>

<details><summary>Hint 3</summary>

```text
depth = pix[:, 2]; uv = pix[:, :2] / depth.clamp(min=1e-5)[:, None]
```

</details>

In [ ]:
def project_points(points: torch.Tensor, K: torch.Tensor, ego_to_cam: torch.Tensor):
    """points [P, 3] in the ego frame -> (uv [P, 2] pixels, depth [P]).

    Divide by the depth clamped to at least 1e-5 so points behind the camera do
    not produce inf (they are marked invalid later anyway).
    """
    raise NotImplementedError  # TODO

## Step 3: BEV pillar points

**Hints for `bev_pillar_points`** (open one at a time):

<details><summary>Hint 1</summary>

```text
x = (torch.arange(bev_h) - (bev_h - 1) / 2) * resolution; same for y with bev_w.
```

</details>

<details><summary>Hint 2</summary>

```text
torch.meshgrid(x, y, indexing="ij") gives [bev_h, bev_w] grids xx, yy.
```

</details>

<details><summary>Hint 3</summary>

```text
stack (xx, yy, z) for every z into [bev_h, bev_w, N_z, 3], then reshape.
```

</details>

In [ ]:
def bev_pillar_points(bev_h: int, bev_w: int, resolution: float,
                      z_anchors=(0.0, 1.0, 2.0, 3.0)) -> torch.Tensor:
    """[bev_h * bev_w, N_z, 3] ego-frame points; cell (i, j) -> query q = i * bev_w + j."""
    raise NotImplementedError  # TODO

## Step 4: reference points and visibility for every camera

**Hints for `reference_points`** (open one at a time):

<details><summary>Hint 1</summary>

```text
loop over cameras and reuse project_points on points.reshape(-1, 3).
```

</details>

<details><summary>Hint 2</summary>

```text
valid = (depth > 1e-5) & (u > 0) & (u < img_w) & (v > 0) & (v < img_h)
```

</details>

<details><summary>Hint 3</summary>

```text
torch.stack the per-camera results and reshape to [N_cam, Q, N_z, ...].
```

</details>

In [ ]:
def reference_points(points: torch.Tensor, Ks: torch.Tensor, ego_to_cams: torch.Tensor,
                     img_h: int, img_w: int):
    """Project pillar points into every camera.

    Args:
        points: [Q, N_z, 3]   Ks: [N_cam, 3, 3]   ego_to_cams: [N_cam, 4, 4]
    Returns:
        uv_norm: [N_cam, Q, N_z, 2]  (2u/W - 1, 2v/H - 1)
        valid:   [N_cam, Q, N_z] bool, depth > 1e-5 and strictly inside the image
    """
    raise NotImplementedError  # TODO

## Step 5: sample image features and fuse them into BEV features

**Hints for `sample_features`** (open one at a time):

<details><summary>Hint 1</summary>

```text
grid_sample takes input [N, C, H_in, W_in] and grid [N, H_out, W_out, 2]
(x first, then y) and returns [N, C, H_out, W_out]; here H_out = Q, W_out = N_z.
```

</details>

<details><summary>Hint 2</summary>

```text
out = F.grid_sample(features, uv_norm, mode="bilinear", padding_mode="zeros",
                    align_corners=False)
```

</details>

<details><summary>Hint 3</summary>

```text
return out.permute(0, 2, 3, 1)
```

</details>

In [ ]:
def sample_features(features: torch.Tensor, uv_norm: torch.Tensor) -> torch.Tensor:
    """Bilinear sampling at the reference points.

    features: [N_cam, C, Hf, Wf]   uv_norm: [N_cam, Q, N_z, 2]
    Returns [N_cam, Q, N_z, C]. Use F.grid_sample(mode="bilinear",
    padding_mode="zeros", align_corners=False): uv_norm treats the image edges
    as -1 and +1, which is the align_corners=False convention.
    """
    raise NotImplementedError  # TODO

**Hints for `aggregate_views`** (open one at a time):

<details><summary>Hint 1</summary>

```text
zero the invalid samples, sum over cameras and heights, divide by the count.
```

</details>

<details><summary>Hint 2</summary>

```text
m = valid[..., None].to(sampled.dtype); total = (sampled * m).sum(dim=(0, 2))
```

</details>

<details><summary>Hint 3</summary>

```text
return total / m.sum(dim=(0, 2)).clamp(min=1)
```

</details>

In [ ]:
def aggregate_views(sampled: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
    """Average the valid samples of each BEV cell over cameras and heights.

    sampled [N_cam, Q, N_z, C], valid [N_cam, Q, N_z] -> bev [Q, C].
    Cells that no camera sees get zeros.
    """
    raise NotImplementedError  # TODO

## Check

Run this after each function you finish. Steps you have not started show TODO; finished steps show PASS. For a full traceback of a failing step: `check.run_lab(9, globals(), verbose=True)`.

In [ ]:
check.run_lab(9, namespace=globals());

## Scratch

Try your functions here: print shapes, call them on small inputs. (Functions you have not written yet raise `NotImplementedError`.)

In [ ]:
# Try the geometry here (Run the cells above first).
K = make_intrinsics(500.0, 800, 448)
T = make_ego_to_cam(0.0, 1.6)
uv, depth = project_points(torch.tensor([[10.0, 2.0, 0.0]]), K, T)
uv, depth                                # guide: (300, 304), depth 10